In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torchvision
from transformers import CLIPProcessor, CLIPTextModelWithProjection
import torch.optim as optim
import pickle
from tqdm import tqdm

/home/rianbutala/projects/sd-v5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("jackyhate/text-to-image-2M", streaming=True)
print(next(iter(ds['train']))['json']['prompt'])
print(next(iter(ds['train']))['jpg'])

A promotional image for a crampon storage bag with a group of hikers in the background. The bag is prominently displayed in the foreground, with a large, detailed image of the bag's mesh design and a smaller image of the bag's orange and blue color scheme. The text 'CRAMPON STORAGE BAG' is displayed at the top, along with the tagline 'Premium Material, Practical Design, Easy to Use'. The background features a snowy mountain landscape with hikers in various colors of jackets and backpacks.
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1024x1024 at 0x79BFF6F1BD40>


In [3]:

transforms = torchvision.transforms.Compose([
  torchvision.transforms.Resize((256, 256)),
  torchvision.transforms.ToTensor()
])

class TextImageDataset(Dataset):
    def __init__(self, dataset, transforms=None):
        self.dataset = dataset
        self.transforms = transforms
        self.dataset_iter = iter(self.dataset)

    def __len__(self):
        return int(1e10)

    def __getitem__(self, idx):
        try:
          item = next(self.dataset_iter)
        except StopIteration:
          self.dataset_iter = iter(self.dataset)
          item = next(self.dataset_iter)
        image = item['jpg'] # PIL image
        prompt = item['json']['prompt'] # text
        image = transforms(image) if self.transforms else image
        return image, prompt

In [4]:
ds_obj = TextImageDataset(ds['train'], transforms=transforms)
dataloader = DataLoader(ds_obj, batch_size=1)
next(iter(dataloader))[0].shape

torch.Size([1, 3, 256, 256])

In [5]:
total_entries = 10000

dl_iter = iter(dataloader)
populate_pickle =False
if populate_pickle:
  with open("data/tensors.pkl", "wb") as f:
    pickler = pickle.Pickler(f, protocol=pickle.HIGHEST_PROTOCOL)
    for i in tqdm(range(total_entries)):
      next_obj = next(dl_iter)
      pickler.dump(next_obj)
      pickler.clear_memo()

In [ ]:
def load_stream(path="data/tensors.pkl"):
    with open(path, "rb") as f:
        while True:
            try:
                # yield unpickler.load()
                yield pickle.Unpickler(f).load()
            except EOFError:
                break

for i in tqdm(load_stream()):
 print(f"\r shape: {i[0].shape}", end="", flush=True)

0it [00:00, ?it/s]


AttributeError: 'list' object has no attribute 'shape'